# DSAI 413 — A2: Notebook 2 — Report Generation (Mode 1)

**Goal:** Run and evaluate the Mode 1 pipeline. Generate structured radiology reports from chest X-rays using MedGemma. Compare prompt variants and compute metrics against ground-truth MIMIC-CXR reports.

**Steps:**
1. Load test images + ground-truth reports
2. Load MedGemma
3. Generate reports using different prompt variants
4. Evaluate with BLEU, ROUGE-L, BERTScore
5. CLIP image-text alignment comparison
6. Qualitative analysis: side-by-side examples

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from src.config import MIMIC_CSV_PATH, MIMIC_IMG_COL, MIMIC_TEXT_COL, EVAL_SAMPLE_SIZE
from src.preprocessing import load_mimic_subset
from src.mode1_report_gen import ReportGenerationPipeline, PROMPT_VARIANTS
from src.evaluation import compute_report_metrics, generate_comparison_table

print('Imports OK')

## 1. Load test data

In [ ]:
with open('../data/split_indices.json') as f:
    splits = json.load(f)

all_images, all_reports, all_paths = load_mimic_subset(
    MIMIC_CSV_PATH, img_col=MIMIC_IMG_COL, text_col=MIMIC_TEXT_COL
)

test_idx = splits['test'][:EVAL_SAMPLE_SIZE]   # cap at eval sample size
test_images  = [all_images[i]  for i in test_idx]
test_reports = [all_reports[i] for i in test_idx]

print(f'Evaluation set: {len(test_images)} images')

## 2. Load pipeline

In [ ]:
pipeline = ReportGenerationPipeline(use_clip=True, medgemma_load_in_4bit=True)
pipeline.load_models()
print('Pipeline ready')

## 3. Generate reports (default prompt)

In [ ]:
# NOTE: This cell may take 10–30 minutes depending on GPU and sample size.
# Reduce EVAL_SAMPLE_SIZE in config.py for faster testing.

results = pipeline.run_batch(test_images, ground_truth_reports=test_reports)

generated_reports = [r['report'] for r in results]
clip_scores       = [r['clip_alignment'] for r in results]

print(f'Generated {len(generated_reports)} reports.')
print(f'Mean CLIP alignment: {sum(s for s in clip_scores if s) / len(clip_scores):.4f}')

## 4. Evaluate: BLEU, ROUGE-L, BERTScore

In [ ]:
metrics = compute_report_metrics(generated_reports, test_reports)
print('\n=== MedGemma (default prompt) ===')    
for k, v in metrics.items():
    print(f'  {k:20s}: {v}')

## 5. Prompt variant comparison (on 5 examples)

In [ ]:
# Compare prompt variants on 5 test images
n_compare = 5
variant_results = {}

for variant_name, prompt_text in PROMPT_VARIANTS.items():
    gen = []
    for img in test_images[:n_compare]:
        report = pipeline._medgemma.generate_report(img, prompt=prompt_text)
        gen.append(report)
    m = compute_report_metrics(gen, test_reports[:n_compare])
    variant_results[variant_name] = m
    print(f'{variant_name}: ROUGE-L={m["rouge_l"]}, BLEU-1={m["bleu1"]}')

# Visualize
variants = list(variant_results.keys())
rouge_l  = [variant_results[v]['rouge_l'] for v in variants]
bleu1    = [variant_results[v]['bleu1']   for v in variants]

x = range(len(variants))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([xi - 0.2 for xi in x], rouge_l, 0.35, label='ROUGE-L', color='#0d6efd')
ax.bar([xi + 0.2 for xi in x], bleu1,   0.35, label='BLEU-1',  color='#28a745')
ax.set_xticks(x)
ax.set_xticklabels(variants, rotation=20)
ax.set_title('Prompt Variant Comparison (MedGemma)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Qualitative examples

In [ ]:
# Show 3 side-by-side examples: image | generated report | ground truth
for i in range(3):
    print(f'\n{'='*80}')
    print(f'Example {i+1}')
    print(f'\n--- GENERATED ---\n{generated_reports[i]}')
    print(f'\n--- GROUND TRUTH ---\n{test_reports[i]}')

    # Show image
    plt.figure(figsize=(4, 4))
    plt.imshow(test_images[i], cmap='gray')
    plt.title(f'X-ray {i+1}')
    plt.axis('off')
    plt.show()

## 7. Build comparison table

In [ ]:
# CLIP zero-shot comparison: embed images → embed GT reports → compute alignment
from src.models.clip_model import CLIPEncoder
import numpy as np

clip_enc = pipeline._clip
if clip_enc:
    img_embs = clip_enc.embed_images(test_images[:20])
    gt_embs  = clip_enc.embed_texts(test_reports[:20])
    gen_embs = clip_enc.embed_texts(generated_reports[:20])

    gt_alignment  = float(np.mean((img_embs * gt_embs).sum(axis=1)))
    gen_alignment = float(np.mean((img_embs * gen_embs).sum(axis=1)))

    model_results = {
        'MedGemma': {
            'task': 'Report Gen',
            **metrics,
            'clip_img_text_sim': round(gen_alignment, 4),
        },
        'GT Reports (CLIP)': {
            'task': 'Reference',
            'clip_img_text_sim': round(gt_alignment, 4),
        },
    }

    table_md = generate_comparison_table(model_results)
    display(Markdown(table_md))